<a href="https://colab.research.google.com/github/BOTTA-GOPINATH/ML_Log_Anomaly_Detector/blob/main/model1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [13]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from sklearn.preprocessing import StandardScaler
data = pd.read_csv("/content/sample_data/Monday-WorkingHours.pcap_ISCX.csv")

print(data.info())


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 529918 entries, 0 to 529917
Data columns (total 79 columns):
 #   Column                        Non-Null Count   Dtype  
---  ------                        --------------   -----  
 0    Destination Port             529918 non-null  int64  
 1    Flow Duration                529918 non-null  int64  
 2    Total Fwd Packets            529918 non-null  int64  
 3    Total Backward Packets       529918 non-null  int64  
 4   Total Length of Fwd Packets   529918 non-null  int64  
 5    Total Length of Bwd Packets  529918 non-null  int64  
 6    Fwd Packet Length Max        529918 non-null  int64  
 7    Fwd Packet Length Min        529918 non-null  int64  
 8    Fwd Packet Length Mean       529918 non-null  float64
 9    Fwd Packet Length Std        529918 non-null  float64
 10  Bwd Packet Length Max         529918 non-null  int64  
 11   Bwd Packet Length Min        529918 non-null  int64  
 12   Bwd Packet Length Mean       529918 non-nul

In [14]:
data.columns = data.columns.str.strip()

data = data.loc[:, ~data.columns.duplicated()]

if 'Fwd Header Length.1' in data.columns:
    data.drop(columns=['Fwd Header Length.1'], inplace=True)

data.replace([np.inf, -np.inf], np.nan, inplace=True)


print("Missing values before:", data.isnull().sum().sum())

data.dropna(inplace=True)

print("Missing values after:", data.isnull().sum().sum())

Missing values before: 874
Missing values after: 0


In [15]:
nunique = data.nunique()
constant_cols = nunique[nunique == 1].index

data.drop(columns=constant_cols, inplace=True)

print("Removed constant columns:", len(constant_cols))
print("Shape after constant removal:", data.shape)

Removed constant columns: 11
Shape after constant removal: (529481, 67)


In [16]:
drop_cols = [
    'Subflow Fwd Packets',
    'Subflow Fwd Bytes',
    'Subflow Bwd Packets',
    'Subflow Bwd Bytes',
    'Fwd Packet Length Min',
    'Bwd Packet Length Max',
    'Bwd Packet Length Min',
    'Fwd Packet Length Max',
    'Fwd Header Length',
    'Bwd Header Length',
    'Fwd PSH Flags',
    'URG Flag Count',
    'ECE Flag Count',
    'Fwd Packet Length Mean',
    'Bwd Packet Length Mean',
    'Fwd Packets/s',
    'Bwd Packets/s',
    'act_data_pkt_fwd',
    'min_seg_size_forward',
    'Avg Fwd Segment Size',
    'Avg Bwd Segment Size',
    'Destination Port'
]

data.drop(columns=drop_cols, inplace=True, errors='ignore')

print("Shape after manual feature removal:", data.shape)
print(data.info())

Shape after manual feature removal: (529481, 45)
<class 'pandas.core.frame.DataFrame'>
Index: 529481 entries, 0 to 529917
Data columns (total 45 columns):
 #   Column                       Non-Null Count   Dtype  
---  ------                       --------------   -----  
 0   Flow Duration                529481 non-null  int64  
 1   Total Fwd Packets            529481 non-null  int64  
 2   Total Backward Packets       529481 non-null  int64  
 3   Total Length of Fwd Packets  529481 non-null  int64  
 4   Total Length of Bwd Packets  529481 non-null  int64  
 5   Fwd Packet Length Std        529481 non-null  float64
 6   Bwd Packet Length Std        529481 non-null  float64
 7   Flow Bytes/s                 529481 non-null  float64
 8   Flow Packets/s               529481 non-null  float64
 9   Flow IAT Mean                529481 non-null  float64
 10  Flow IAT Std                 529481 non-null  float64
 11  Flow IAT Max                 529481 non-null  int64  
 12  Flow IAT Min  

In [21]:
print("Original Shape:", data.shape)

scaler = StandardScaler()

X_scaled = scaler.fit_transform(data)

X_scaled = pd.DataFrame(X_scaled, columns=data.columns)

print("Scaled Shape:", X_scaled.shape)

Original Shape: (529481, 45)
Scaled Shape: (529481, 45)


In [22]:
baseline_stats = pd.DataFrame({
    "Mean": X_scaled.mean(),
    "Std": X_scaled.std(),
    "Median": X_scaled.median(),
    "P25": X_scaled.quantile(0.25),
    "P75": X_scaled.quantile(0.75),
    "P95": X_scaled.quantile(0.95),
    "P99": X_scaled.quantile(0.99)
})

Q1 = X_scaled.quantile(0.25)
Q3 = X_scaled.quantile(0.75)

IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

In [23]:
from scipy.spatial.distance import mahalanobis
from numpy.linalg import inv

# Covariance matrix
cov_matrix = np.cov(X_scaled.T)

# Inverse covariance
inv_cov_matrix = inv(cov_matrix)

# Mean vector
mean_vector = X_scaled.mean().values


def mahalanobis_distance(row):
    return mahalanobis(row, mean_vector, inv_cov_matrix)


print("Calculating anomaly score...")

X_scaled["anomaly_score"] = X_scaled.apply(mahalanobis_distance, axis=1)

X_scaled["anomaly_score"].describe()

Calculating anomaly score...


,anomaly_score
count,529481.000000
mean,4.320208
std,5.131838
min,1.254719
25%,1.783249
50%,2.919870
75%,4.792475
max,674.798769


In [24]:
threshold = X_scaled["anomaly_score"].quantile(0.95)

X_scaled["anomaly_label"] = (X_scaled["anomaly_score"] > threshold).astype(int)

print(X_scaled["anomaly_label"].value_counts())

anomaly_label
0    503007
1     26474
Name: count, dtype: int64
